# P6 -- Agentes y RAG con Indexacion Jerarquica

**RA/CE**: RA3e (toma de decisiones basada en datos)
**Fase**: F6 -- Agentes y RAG en Produccion
**Teoria asociada**: `01-teoria/07_agentes_rag.md`

---

## Objetivos de Aprendizaje

1. Comprender las limitaciones del RAG plano frente a documentos jerarquicos
2. Implementar indexacion jerarquica con LlamaIndex
3. Disenar un sistema multi-agente con CrewAI sobre un indice jerarquico
4. Evaluar como multi-agente + jerarquia mejora la toma de decisiones (RA3e)

## Prerrequisitos

- RAG basico con LangChain (visto en UD6)
- Concepto de agente y tool (visto en UD6)
- CrewAI instalado: pip install crewai
- LlamaIndex instalado: pip install llama-index
- HuggingFace embeddings: pip install llama-index-embeddings-huggingface

## Contexto: ParkingCorp

Trabajaras con **ParkingCorp**, empresa que gestiona multiples aparcamientos.
Su documentacion esta organizada jerarquicamente: Empresa -> Parking -> Cliente ->
Vehiculo -> Transaccion. Construiras un sistema multi-agente que navega esta
jerarquia para responder consultas de clientes, operarios y administradores.


---
## Parte 0: Recordatorio -- RAG tradicional

En la unidad **Modelos de la IA** (UD6) trabajaste el RAG con ChromaDB + embeddings:
fragmentar documentos en chunks, embedding, top-k por similitud coseno, respuesta.

Ese enfoque es correcto para knowledge bases planas. En esta practica vamos un paso
mas alla: documentos con **jerarquia** (empresa, parking, cliente, vehiculo)
donde el RAG plano pierde contexto.

Si no has hecho la practica de RAG basico de UD6, revisala primero.
Aqui asumimos que comprendes: embeddings, vector store, busqueda por similitud,
y el patron recuperacion-generacion.


---
## Parte 1: Verifica tu entorno


In [ ]:
import sys, os, json

HAS_DEPS = True
missing = []

# Core: LlamaIndex
try:
    from llama_index.core import Document, Settings
    from llama_index.core.node_parser import HierarchicalNodeParser
    from llama_index.core.retrievers import RecursiveRetriever
    from llama_index.core.indices.vector_store import VectorStoreIndex
    print('OK LlamaIndex core')
except ImportError:
    HAS_DEPS = False
    missing.append('llama-index')
    print('KO Missing: llama-index')

# Embeddings
try:
    from llama_index.embeddings.huggingface import HuggingFaceEmbedding
    print('OK HuggingFace embeddings')
except ImportError:
    HAS_DEPS = False
    missing.append('llama-index-embeddings-huggingface')
    print('KO Missing: llama-index-embeddings-huggingface')

# CrewAI
try:
    from crewai import Agent, Task, Crew, Process
    from crewai.tools import BaseTool
    print('OK CrewAI')
except ImportError:
    HAS_DEPS = False
    missing.append('crewai')
    print('KO Missing: crewai')

# Data
try:
    import pandas as pd
    import numpy as np
    print('OK Data dependencies')
except ImportError as e:
    HAS_DEPS = False
    missing.append('pandas/numpy')
    print('KO Missing: data libs')

if not HAS_DEPS:
    print(f'Install: pip install {" ".join(missing)}')
else:
    print('Environment ready')


---
## Parte 2: Documentos jerarquicos de ParkingCorp

Vamos a crear la base documental de ParkingCorp con estructura jerarquica.
Cada nivel contiene informacion que depende del nivel superior:

| Nivel | Contenido | Ejemplo |
|-------|-----------|---------|
| **1. Empresa** | Politicas corporativas, tarifas globales | ParkingCorp gestiona 23 parkings... |
| **2. Parking** | Datos especificos de cada parking | Parking Norte, Calle Mayor 15... |
| **3. Cliente** | Abonos, contratos, datos de pago | Cliente A, Premium Plus 120E/mes... |
| **4. Vehiculo** | Matriculas, tipos, incidencias | 1234ABC, turismo... |
| **5. Transaccion** | Pagos, multas, disputas | Pago 15/06, 5E, 2h... |


In [ ]:
# --- Documentos jerarquicos de ParkingCorp ---
# Usamos diccionarios con claves string para mantener la jerarquia

docs = {
    'empresa': {
        'titulo': 'ParkingCorp - Politicas Corporativas',
        'nivel': 1,
        'contenido': '''ParkingCorp gestiona 23 aparcamientos en la ciudad.
Facturacion anual: 12ME.

TARIFAS GENERALES:
- Tarifa base: 2E/hora en parking estandar
- Tarifa premium: 3E/hora en parking premium
- Tarifa nocturna (22:00-07:00): 5E fijo
- Perdida de ticket: recargo de 20E

HORARIOS:
- Todos los parkings abren de 7:00 a 23:00
- Parking Sur: horario ampliado 24h

ABONOS:
- Basico: 80E/mes, 1 vehiculo
- Premium: 120E/mes, 2 vehiculos
- Premium Plus: 150E/mes, 3 vehiculos
- Trimestral Basico: 210E/3 meses

POLITICA DE MULTAS:
- Estacionamiento sin pago: 30E
- Exceso de tiempo: 10E por hora extra
- Las disputas se resuelven en 72h habiles'''
    },
    'parking_norte': {
        'titulo': 'Parking Norte',
        'nivel': 2, 'padre': 'empresa',
        'contenido': '''Parking Norte - Calle Mayor 15
Capacidad: 200 plazas (10 motos, 5 VE)
Tarifa: 2,50E/hora (recargo centrico)
Abonos mensuales: Basico 95E, Premium 135E
Plazas reservadas para abonados: 50
Servicios: vigilancia 24h, lavado basico'''
    },
    'parking_sur': {
        'titulo': 'Parking Sur',
        'nivel': 2, 'padre': 'empresa',
        'contenido': '''Parking Sur - Avda. del Puerto 42
Capacidad: 350 plazas (20 motos, 10 VE)
Tarifa: 1,80E/hora
Tarifa plana nocturna: 5E (22:00-07:00)
Abonos: Basico 75E, Premium 110E
Horario: 24h (unico con horario ampliado)
Servicios: vigilancia 24h, lavado completo, cafeteria'''
    },
    'cliente_a': {
        'titulo': 'Cliente A - Maria Garcia',
        'nivel': 3, 'padre': 'parking_norte',
        'contenido': '''Cliente: Maria Garcia Lopez
Plan: Premium Plus - 150E/mes
Vehiculos permitidos: 3
Forma de pago: domiciliacion bancaria
Telefono: +34 611 222 333
Email: maria.garcia@email.com
Fecha de alta: 01/03/2025
Parking asignado: Parking Norte'''
    },
    'cliente_b': {
        'titulo': 'Cliente B - Carlos Lopez',
        'nivel': 3, 'padre': 'parking_sur',
        'contenido': '''Cliente: Carlos Lopez Martinez
Plan: Trimestral Basico - 210E/trimestre
Vehiculos permitidos: 1
Forma de pago: tarjeta de credito
Telefono: +34 622 333 444
Fecha de alta: 10/11/2025
Parking asignado: Parking Sur'''
    },
    'vehiculo_x': {
        'titulo': 'Vehiculo 1234ABC - Maria Garcia',
        'nivel': 4, 'padre': 'cliente_a',
        'contenido': '''Matricula: 1234ABC
Tipo: Turismo (Sedan)
Marca: Toyota Corolla - Color: Blanco
Abono: Premium Plus
Ultimo acceso: 18/06/2026 09:15
Ultima incidencia: Disputa de multa - 10/06/2026'''
    },
    'vehiculo_y': {
        'titulo': 'Vehiculo 5678DEF - Maria Garcia',
        'nivel': 4, 'padre': 'cliente_a',
        'contenido': '''Matricula: 5678DEF
Tipo: Motocicleta
Marca: Honda PCX125 - Color: Rojo
Abono: Premium Plus (segundo vehiculo)
Ultimo acceso: 17/06/2026 14:30'''
    },
    'transaccion_pago': {
        'titulo': 'Pago 15/06/2026 - Parking Norte',
        'nivel': 5, 'padre': 'vehiculo_x',
        'contenido': '''Transaccion: PAGO-2026-0615-001
Fecha: 15/06/2026 | Parking: Norte
Matricula: 1234ABC | Tiempo: 2h
Importe: 5,00E | Metodo: Tarjeta
Estado: Completado'''
    },
    'transaccion_multa': {
        'titulo': 'Disputa de multa - 1234ABC',
        'nivel': 5, 'padre': 'vehiculo_x',
        'contenido': '''Transaccion: MULTA-2026-0610-003
Fecha: 10/06/2026 | Parking: Norte
Matricula: 1234ABC
Motivo: Exceso de tiempo (3h extra)
Importe: 30,00E
Estado: DISPUTADA
Cliente: Maria Garcia (apelacion iniciada)
Notas: El cliente alega que el sistema no registro su salida'''
    },
}

print(f'OK {len(docs)} documentos jerarquicos creados')
for k, v in docs.items():
    print(f'   [N{v["nivel"]}] {v["titulo"]}')


---
## Parte 3: Indexacion jerarquica con LlamaIndex

Vamos a construir un indice jerarquico usando `HierarchicalNodeParser`.
El parser organiza el texto en una jerarquia de nodos con diferentes
tamanos de chunk: grandes para contexto global, pequenos para detalle.

Luego usaremos `RecursiveRetriever` para navegar esta jerarquia:
1. Encuentra los nodos hoja mas relevantes
2. Sube al nodo padre si necesita mas contexto
3. Baja a nodos hermanos si la consulta requiere granularidad fina


In [ ]:
from llama_index.core import Document, Settings
from llama_index.core.node_parser import HierarchicalNodeParser
from llama_index.core.indices.vector_store import VectorStoreIndex
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# Configurar embeddings locales
Settings.embed_model = HuggingFaceEmbedding(
    model_name='all-MiniLM-L6-v2',
    embed_batch_size=10
)
Settings.chunk_size = 2048
Settings.chunk_overlap = 50

# Convertir documentos a objetos Document de LlamaIndex
docs_llama = []
for key, doc in docs.items():
    d = Document(
        text=doc['contenido'],
        metadata={
            'id': key,
            'titulo': doc['titulo'],
            'nivel': doc['nivel'],
            'padre': doc.get('padre', '')
        }
    )
    docs_llama.append(d)

print(f'OK {len(docs_llama)} documentos cargados en LlamaIndex')
for d in docs_llama:
    t = d.metadata['titulo']
    n = d.metadata['nivel']
    print(f'   [N{n}] {t} - {len(d.text)} chars')


In [ ]:
# Crear el parser jerarquico
# chunk_sizes define la profundidad: [2048, 512, 128]
# - Nivel 0: 2048 chars -> contexto de empresa/parking
# - Nivel 1: 512 chars  -> contexto de cliente/vehiculo
# - Nivel 2: 128 chars  -> detalle especifico (tarifas, fechas)
parser = HierarchicalNodeParser.from_defaults(
    chunk_sizes=[2048, 512, 128],
    chunk_overlap=20
)

# Obtener todos los nodos jerarquicos
nodes = parser.get_nodes_from_documents(docs_llama)

print(f'OK Arbol jerarquico creado: {len(nodes)} nodos totales')

# Mostrar distribucion por nivel
from collections import Counter
niveles = Counter(n.node_id.count('_') for n in nodes)
print(f'   Distribucion de profundidad: {dict(sorted(niveles.items()))}')

# Mostrar algunos ejemplos
print('Ejemplos de nodos:')
for n in nodes[:4]:
    pid = n.ref_doc_id[:30] if n.ref_doc_id else 'N/A'
    print(f'   Nodo: {n.node_id[:40]}...')
    print(f'   Texto: {n.text[:80]}...')
    print(f'   Padre: {pid}...')
    print()


In [ ]:
# Crear el indice vectorial sobre los nodos jerarquicos
index = VectorStoreIndex(nodes, embed_model=Settings.embed_model)

# Configurar el RecursiveRetriever
from llama_index.core.retrievers import RecursiveRetriever

recursive_retriever = RecursiveRetriever(
    retriever_dict={
        'vector': index.as_retriever(similarity_top_k=5)
    },
    node_parser=parser,
    verbose=True
)

print('OK RecursiveRetriever configurado')
print('   similarity_top_k: 5')
print('   Niveles jerarquicos: 3 (2048 / 512 / 128 chars)')


In [ ]:
# Probar el recuperador jerarquico con varias consultas

consultas = [
    'Cuanto paga Maria Garcia por hora en Parking Norte?',
    'Puede el cliente A acceder al Parking Centro con su abono?',
    'Que paso con la multa del vehiculo 1234ABC?',
    'Cual es la tarifa nocturna del Parking Sur?',
]

for q in consultas:
    print(f'\n{"="*60}')
    print(f'Consulta: {q}')
    print('='*60)

    nodes_recuperados = recursive_retriever.retrieve(q)

    for i, n in enumerate(nodes_recuperados[:3]):
        titulo = n.metadata.get('titulo', 'Sin titulo')
        nivel = n.metadata.get('nivel', '?')
        score = n.score if hasattr(n, 'score') else 'N/A'
        if isinstance(score, float):
            print(f'  [{i+1}] {titulo} (nivel={nivel}, score={score:.3f})')
        else:
            print(f'  [{i+1}] {titulo} (nivel={nivel})')
        print(f'      {n.text[:150].strip()}...')


---
## Parte 4: Sistema multi-agente con CrewAI

Ahora combinamos la indexacion jerarquica con un sistema multi-agente.
Cada agente tiene un rol especifico en el flujo de consulta del parking:

| Agente | Rol | Responsabilidad |
|--------|-----|-----------------|
| **Clasificador** | Analiza la consulta | Determina que nivel jerarquico consultar |
| **Recuperador** | Navega el indice jerarquico | Usa RecursiveRetriever para info precisa |
| **Generador** | Produce la respuesta | Redacta respuesta clara citando fuentes |
| **Validador** | Verifica contra fuentes | Comprueba que cada afirmacion esta respaldada |


In [ ]:
from crewai import Agent, Task, Crew, Process
from crewai.tools import BaseTool

# ---- Herramienta 1: Recuperacion jerarquica ----
class HierarchicalRetrieverTool(BaseTool):
    name: str = 'Recuperador Jerarquico'
    description: str = (
        'Busca informacion en la base documental jerarquica de ParkingCorp. '
        'Niveles: empresa (politicas), parking (tarifas), '
        'cliente (abonos), vehiculo (matriculas), transaccion (pagos).'
    )

    def _run(self, query: str, nivel: str = 'auto') -> str:
        nodes = recursive_retriever.retrieve(query)
        if not nodes:
            return 'No se encontro informacion relevante.'
        output_lines = []
        for n in nodes[:5]:
            nd = n.metadata.get('nivel', '?')
            t = n.metadata.get('titulo', 'Documento')
            output_lines.append(f'[Nivel {nd}] {t}\n{n.text.strip()}\n')
        return '\n---\n'.join(output_lines)

# ---- Herramienta 2: Clasificacion de consulta ----
class QueryClassifierTool(BaseTool):
    name: str = 'Clasificador de Consultas'
    description: 'Determina que nivel de la jerarquia necesita la consulta.'

    def _run(self, query: str) -> str:
        q = query.lower()
        if any(w in q for w in ['politica', 'corporativo', 'global', 'general']):
            return 'empresa'
        elif any(w in q for w in ['parking', 'aparcamiento', 'plaza', 'norte', 'sur']):
            return 'parking'
        elif any(w in q for w in ['cliente', 'abono', 'plan', 'mensual', 'maria', 'carlos']):
            return 'cliente'
        elif any(w in q for w in ['vehiculo', 'matricula', 'coche', 'moto', '1234', '5678']):
            return 'vehiculo'
        elif any(w in q for w in ['multa', 'pago', 'transaccion', 'disputa', 'incidencia']):
            return 'transaccion'
        else:
            return 'auto'

hierarchical_tool = HierarchicalRetrieverTool()
classifier_tool = QueryClassifierTool()

print('OK Herramientas definidas:')
print('   1. Recuperador Jerarquico (usa RecursiveRetriever)')
print('   2. Clasificador de Consultas (por palabras clave)')

# Probar clasificacion
tests = [
    'Cuanto cuesta el abono mensual en Parking Norte?',
    'El cliente Maria Garcia tiene una disputa de multa',
    'Cuales son las politicas generales de ParkingCorp?',
]
print('Prueba de clasificacion:')
for t in tests:
    nivel = classifier_tool._run(t)
    print(f'  {t[:50]}... -> {nivel}')


In [ ]:
# ---- Agente 1: Clasificador ----
clasificador = Agent(
    role='Clasificador de Consultas de ParkingCorp',
    goal='Determinar que nivel de la jerarquia necesita la consulta',
    backstory=(
        'Experto en gestion de aparcamientos. '
        'Identifica si una pregunta es sobre politicas corporativas, '
        'un parking, un cliente, un vehiculo o una transaccion.'
    ),
    tools=[classifier_tool],
    verbose=True,
    allow_delegation=False
)

# ---- Agente 2: Recuperador jerarquico ----
recuperador = Agent(
    role='Recuperador Jerarquico',
    goal='Navegar el arbol documental de ParkingCorp para encontrar informacion precisa',
    backstory=(
        'Especialista en busqueda multinivel. '
        'Sabe navegar de empresa a parking, de parking a cliente, '
        'y de cliente a vehiculo segun la consulta.'
    ),
    tools=[hierarchical_tool],
    verbose=True,
    allow_delegation=False
)

# ---- Agente 3: Generador de respuesta ----
generador = Agent(
    role='Generador de Respuesta de ParkingCorp',
    goal='Producir una respuesta clara usando el contexto recuperado',
    backstory=(
        'Redactor tecnico con experiencia en atencion al cliente '
        'de aparcamientos. Explica tarifas, politicas e incidencias '
        'de forma comprensible.'
    ),
    verbose=True,
    allow_delegation=False
)

# ---- Agente 4: Validador ----
validador = Agent(
    role='Validador de Respuestas de ParkingCorp',
    goal='Verificar que la respuesta es correcta segun las fuentes documentales',
    backstory=(
        'Auditor senior de ParkingCorp. '
        'Comprueba que cada afirmacion de la respuesta '
        'este respaldada por la documentacion del parking.'
    ),
    tools=[hierarchical_tool],
    verbose=True,
    allow_delegation=False
)

print('OK 4 agentes definidos:')
print('   1. Clasificador de Consultas')
print('   2. Recuperador Jerarquico')
print('   3. Generador de Respuesta')
print('   4. Validador de Respuestas')


In [ ]:
# ---- Tarea 1: Clasificar la consulta ----
t1_clasificar = Task(
    description=(
        'Analiza la siguiente consulta sobre ParkingCorp:\n\n'
        "'{consulta}'\n\n"
        'Determina a que nivel de la jerarquia pertenece:\n'
        '- empresa: politicas corporativas\n'
        '- parking: datos de un aparcamiento\n'
        '- cliente: abonos y contratos\n'
        '- vehiculo: matriculas y tipos\n'
        '- transaccion: pagos e incidencias\n'
        'Proporciona el nivel y una breve justificacion.'
    ),
    agent=clasificador,
    expected_output='Nivel identificado mas justificacion'
)

# ---- Tarea 2: Recuperar informacion jerarquica ----
t2_recuperar = Task(
    description=(
        'Dada la consulta:\n\n'
        "'{consulta}'\n\n"
        'Busca en el indice jerarquico de ParkingCorp la info relevante.\n'
        'Usa el Recuperador Jerarquico para navegar el arbol documental.\n'
        'Proporciona los docs encontrados con su nivel y contenido.'
    ),
    agent=recuperador,
    expected_output='Documentos por nivel jerarquico'
)

# ---- Tarea 3: Generar respuesta ----
t3_generar = Task(
    description=(
        'Con la informacion recuperada, genera una respuesta clara\n'
        'para la consulta:\n\n'
        "'{consulta}'\n\n"
        'La respuesta debe:\n'
        '1. Responder directamente a la pregunta\n'
        '2. Citar las fuentes documentales usadas\n'
        '3. Ser comprensible para un usuario no tecnico\n'
        '4. Indicar el nivel jerarquico de cada dato'
    ),
    agent=generador,
    expected_output='Respuesta estructurada citando fuentes'
)

# ---- Tarea 4: Validar la respuesta ----
t4_validar = Task(
    description=(
        'Revisa la respuesta generada para:\n\n'
        "'{consulta}'\n\n"
        'Verifica que:\n'
        '1. Cada afirmacion esta respaldada por las fuentes\n'
        '2. Los niveles jerarquicos son correctos\n'
        '3. No hay informacion inventada o contradictoria\n'
        'Proporciona: VALIDA, INCOMPLETA o INCORRECTA con evidencia.'
    ),
    agent=validador,
    expected_output='Validacion con verifiacion de fuentes'
)

print('OK 4 tareas (clasificar -> recuperar -> generar -> validar)')


In [ ]:
# Ensamblar el crew
crew = Crew(
    agents=[clasificador, recuperador, generador, validador],
    tasks=[t1_clasificar, t2_recuperar, t3_generar, t4_validar],
    process=Process.sequential,
    verbose=True
)

# Consultas de ejemplo sobre ParkingCorp
consultas_parking = [
    'Puede Maria Garcia acceder al Parking Centro con su abono Premium Plus?',
    'Cuanto costo la multa del vehiculo 1234ABC y por que esta en disputa?',
    'Que tarifas tiene el Parking Sur para clientes sin abono?',
]

print(f'OK Crew listo. Consultas: {len(consultas_parking)}')
for i, c in enumerate(consultas_parking):
    print(f'  [{i}] {c[:70]}...')


In [ ]:
# Seleccionar consulta y ejecutar el crew
consulta_idx = 0  # Cambia a 1 o 2 para probar otras consultas
consulta = consultas_parking[consulta_idx]

print(f'Consulta:\n{consulta}\n')
print('='*60)
print('INICIANDO FLUJO MULTI-AGENTE SOBRE INDICE JERARQUICO')
print('='*60)

# Ejecutar el crew
resultado = crew.kickoff(inputs={'consulta': consulta})

print('\n' + '='*60)
print('RESPUESTA FINAL')
print('='*60)
print(resultado)


---
## Parte 5: Evaluacion del sistema multi-agente (RA3e)

### 5.1 Analisis del resultado

Analiza como se comporto el sistema multi-agente con la consulta:


In [ ]:
# Tu analisis aqui

analisis = {
    'consulta': consulta,
    'nivel_correcto': None,
    'recuperacion_util': None,
    'respuesta_completa': None,
    'validacion_correcta': None,
    'mejoras': []
}

# Completa el analisis:
# analisis['nivel_correcto'] = True/False
# analisis['recuperacion_util'] = True/False
# analisis['respuesta_completa'] = True/False
# analisis['validacion_correcta'] = True/False
# analisis['mejoras'].append('...')

print('=== ANALISIS DEL SISTEMA MULTI-AGENTE ===')
for k, v in analisis.items():
    print(f'  {k}: {v}')


### 5.2 Comparacion: RAG plano vs. jerarquico multi-agente (RA3e)

Completa la tabla comparativa:


In [ ]:
import pandas as pd

tabla = pd.DataFrame([
    {
        'Arquitectura': 'RAG plano (ChromaDB + embeddings)',
        'Recuperacion': 'Top-k chunks planos',
        'Contexto': 'Ninguno - cada chunk es independiente',
        'Casos parking': 'Consultas simples de un solo nivel'
    },
    {
        'Arquitectura': 'Indexacion jerarquica (LlamaIndex)',
        'Recuperacion': 'Navegacion padre -> hijo -> hermano',
        'Contexto': 'Preserva relaciones jerarquicas completas',
        'Casos parking': 'Consultas multi-nivel con relaciones'
    },
    {
        'Arquitectura': 'Jerarquico + Multi-agente (CrewAI)',
        'Recuperacion': 'Clasificacion + navegacion + validacion',
        'Contexto': 'Cada agente aporta perspectiva diferente',
        'Casos parking': 'Consultas complejas con decision y validacion'
    },
])

print(tabla.to_string(index=False))

print('PREGUNTA RA3e:')
print('Como mejora la combinacion de indexacion jerarquica')
print('+ multi-agente la toma de decisiones en ParkingCorp?')
print('En que casos el RAG plano seria suficiente?')


---
## Parte 6: Ejercicios

### Ejercicio 1: Agente especializado por nivel

Anade un quinto agente `EspecialistaTarifas` que se enfoque exclusivamente
en consultas sobre tarifas y precios. Debe usar una herramienta que filtre
la recuperacion jerarquica al nivel de tarifas.


In [ ]:
# TODO: Crea un agente EspecialistaTarifas con su propia herramienta
# Pista: define una subclase de BaseTool que filtre por nivel

# class TarifaFilterTool(BaseTool):
#     name = 'Consultar Tarifas'
#     description = '...'
#     def _run(self, query):
#         # Usa recursive_retriever pero filtra por nivel parking
#         pass

# Tu codigo aqui


### Ejercicio 2: Memoria de consultas

Implementa un mecanismo de memoria simple para que el sistema recuerde
consultas anteriores y no repita recuperacion jerarquica para preguntas
similares.


In [ ]:
# TODO: Implementa una cache simple de consultas resueltas

cache_consultas = {}

def consultar_con_cache(consulta: str) -> str:
    '''Busca en cache o ejecuta recuperacion jerarquica.'''
    # Tu codigo aqui
    pass

# Prueba
# resultado = consultar_con_cache('Cuanto paga Maria Garcia?')
# print(resultado)


### Ejercicio 3: Evaluacion RA3e

Analiza el caso ParkingCorp desde la perspectiva de toma de decisiones (RA3e):

1. Que decisiones puede tomar ParkingCorp gracias a la combinacion de
   indexacion jerarquica + multi-agente que NO podria con RAG plano?
2. En que situaciones un agente unico con RAG plano seria preferible?
3. Como afecta la calidad de la respuesta a la toma de decisiones del negocio?


In [ ]:
# Escribe tu analisis aqui

reflexion_ra3e = {
    'decisiones_solo_con_jerarquia': [],
    'casos_rag_plano_suficiente': [],
    'impacto_en_negocio': ''
}

# Completa:
# reflexion_ra3e['decisiones_solo_con_jerarquia'].append('...')
# reflexion_ra3e['casos_rag_plano_suficiente'].append('...')
# reflexion_ra3e['impacto_en_negocio'] = '...'

print('=== REFLEXION RA3e ===')
print('Decisiones solo con jerarquia:')
for d in reflexion_ra3e['decisiones_solo_con_jerarquia']:
    print(f'  - {d}')
print('Casos donde RAG plano es suficiente:')
for c in reflexion_ra3e['casos_rag_plano_suficiente']:
    print(f'  - {c}')
print(f"Impacto: {reflexion_ra3e['impacto_en_negocio']}")


---
## Bonus: AutoGen

Si quieres explorar el patron **Debate** con AutoGen,
revisa el Apendice en `01-teoria/07_agentes_rag.md`.

Alli encontraras un ejemplo de como dos agentes (defensor del cliente
y defensor de la empresa) debaten una disputa de multa con moderador.

---
## Resumen de la Practica

OK Has comprendido las limitaciones del RAG plano para docs jerarquicos
OK Has implementado indexacion jerarquica con LlamaIndex
OK Has construido un sistema multi-agente con CrewAI sobre indice jerarquico
OK Has evaluado como multi-agente + jerarquia mejora la toma de decisiones
OK Has comparado RAG plano vs. jerarquico vs. jerarquico multi-agente
OK Has analizado el caso ParkingCorp desde la perspectiva de negocio

**Bonus completado**: Lee el apendice de AutoGen en la teoria
